## 1) Downloads, Imports, etc.

In [ ]:
!pip install rdkit
!pip install torch
!pip install torchani
!pip install pennylane
!pip install requests aiohttp
!pip install bokeh
import pennylane as qml

# pip install rdkit-pypi torch torchani (if using ANI)
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np

### powell optimizer works great - conjugate gradient options, from Cody
### bayesian optimization - from Cody


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## 2) Define a whole bunch of steps to prepare the molecules from SMILES, compute ani energies, etc. etc.


In [23]:
MILES = "O"  # Water
NAME = "water"

NCONF = 150                 # more for larger side chains
RMS_PRUNE = 0.4             # Å
KEEP_MMFF = 50              # keep this many lowest by MMFF energy
USE_ANI = True             # set True to compute ANI-2x energies here

def prepare_mol(smiles):
    m = Chem.MolFromSmiles(smiles)
    m = Chem.AddHs(m)
    return m

def embed_minimize_confs(mol, nconf=NCONF):
    params = AllChem.ETKDGv3()
    params.pruneRmsThresh = -1.0   # we’ll prune ourselves
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=nconf, params=params)
    # MMFF minimize
    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant='MMFF94s')
    e_list = []
    for cid in conf_ids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize(maxIts=500)
            e = ff.CalcEnergy()
        except Exception:
            e = 1e9
        e_list.append((cid, e))
    return e_list

def prune_by_rmsd(mol, conf_ids, rms_cut=RMS_PRUNE):
    kept = []
    for cid in conf_ids:
        keep = True
        for kc in kept:
            rms = rdMolAlign.GetBestRMS(mol, mol, prbId=cid, refId=kc)
            if rms < rms_cut:
                keep = False
                break
        if keep:
            kept.append(cid)
    return kept

def mmff_rank_and_prune(mol, conf_energy_pairs, keep=KEEP_MMFF):
    conf_energy_pairs = sorted(conf_energy_pairs, key=lambda x: x[1])
    # Take top 'keep' by energy but ensure diversity with RMSD pruning
    ranked = [cid for cid,_ in conf_energy_pairs]
    diverse = prune_by_rmsd(mol, ranked, rms_cut=RMS_PRUNE)
    # keep the best among those diverse; if too many, cap at 'keep'
    diverse_sorted = sorted(diverse, key=lambda cid: dict(conf_energy_pairs)[cid])
    return diverse_sorted[:keep]

def compute_ani_energies(mol, conf_ids, model_name='ani2x'):
    import torch, torchani
    # Load model
    model = (torchani.models.ANI2x() if model_name.lower() == 'ani2x'
             else torchani.models.ANI1ccx())
    device = torch.device('cpu')
    model = model.to(device).eval()

    # Map atomic numbers -> element symbols for TorchANI
    z2sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F', 16:'S', 17:'Cl', 35:'Br', 53:'I'}
    symbols = [z2sym[atom.GetAtomicNum()] for atom in mol.GetAtoms()]

    # TorchANI helper to build species tensor
    species = model.consts.species_to_tensor(symbols).unsqueeze(0).to(device)  # shape (1, natoms)

    energies = {}
    for cid in conf_ids:
        conf = mol.GetConformer(cid)
        coords = [[conf.GetAtomPosition(i).x,
                   conf.GetAtomPosition(i).y,
                   conf.GetAtomPosition(i).z] for i in range(mol.GetNumAtoms())]
        coordinates = torch.tensor([coords], dtype=torch.float32, device=device)  # (1, natoms, 3)
        with torch.no_grad():
            e = model((species, coordinates)).energies.item()  # Hartree
        energies[cid] = e
    return energies  # dict: confId -> Eh


def write_sdf(mol, conf_ids, fields, path):
    w = Chem.SDWriter(path)
    for cid in conf_ids:
        m = Chem.Mol(mol)
        m.SetProp("_Name", f"{NAME}_conf{cid}")
        for k,v in fields.items():
            if cid in v:
                m.SetDoubleProp(k, float(v[cid]))
        w.write(m, confId=cid)
    w.close()




### 3) Test for water

In [24]:
##########################
### Test for h2o)
##########################
mol0 = prepare_mol(MILES)
tauts = [mol0]

print(f"Found {len(tauts)} unique tautomers")
best_overall = None  # (energy_Eh, taut_idx, conf_id)

for i, taut in enumerate(tauts):
    confEs = embed_minimize_confs(taut, NCONF)
    keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)

    aniE = {}
    if USE_ANI:
        aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')

    # Report for this tautomer
    if USE_ANI and aniE:
        cid_min = min(aniE, key=lambda k: aniE[k])
        E_min = aniE[cid_min]   # Hartree
        print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
        if (best_overall is None) or (E_min < best_overall[0]):
            best_overall = (E_min, i, cid_min)
    else:
        # fall back to MMFF (NOT electronic) just so something prints
        mmffE = dict(confEs)
        cid_min = min(keep_ids, key=lambda k: mmffE[k])
        print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")

if USE_ANI and best_overall:
    E, ti, ci = best_overall
    print(f"\nGround-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")

Found 1 unique tautomers
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -76.388251390 Eh  (conf 81)

Ground-state estimate (ANI): -76.388251390 Eh  from tautomer 0, conformer 81


### 4) estimate for molecules.

In [25]:
def estimate_gse_for_molecules(smiles_list, names, out_file="estimations_gse.txt"):
    with open(out_file, "w") as f:
        for SMILES, NAME in zip(smiles_list, names):
            print(f"\nProcessing: {NAME} ({SMILES})")
            mol0 = prepare_mol(SMILES)
            tauts = [mol0]
            best_overall = None  # (energy_Eh, taut_idx, conf_id)
            for i, taut in enumerate(tauts):
                confEs = embed_minimize_confs(taut, NCONF)
                keep_ids = mmff_rank_and_prune(taut, confEs, keep=KEEP_MMFF)
                aniE = {}
                if USE_ANI:
                    aniE = compute_ani_energies(taut, keep_ids, model_name='ani2x')
                if USE_ANI and aniE:
                    cid_min = min(aniE, key=lambda k: aniE[k])
                    E_min = aniE[cid_min]   # Hartree
                    print(f"[taut {i}] lowest ANI energy: {E_min:.9f} Eh  (conf {cid_min})")
                    if (best_overall is None) or (E_min < best_overall[0]):
                        best_overall = (E_min, i, cid_min)
                else:
                    mmffE = dict(confEs)
                    cid_min = min(keep_ids, key=lambda k: mmffE[k])
                    print(f"[taut {i}] lowest MMFF (not electronic): {mmffE[cid_min]:.3f} kcal/mol  (conf {cid_min})")
            if USE_ANI and best_overall:
                E, ti, ci = best_overall
                result = f"{NAME}\t{SMILES}\t{E:.9f} Eh\tTautomer {ti}\tConformer {ci}\n"
                print(f"Ground-state estimate (ANI): {E:.9f} Eh  from tautomer {ti}, conformer {ci}")
                f.write(result)
            else:
                result = f"{NAME}\t{SMILES}\tNo ANI result\n"
                f.write(result)

# Example usage:
smiles_list = [
    "O",  # Water
    "N[C@@H](C)C(=O)O",         # Alanine
    "CC(C)C(N)C(=O)O",          # Valine
    "CC(C)CC(N)C(=O)O",         # Leucine
    "CC(C)C(N)C(=O)O",          # Isoleucine
    "C(C(C(=O)O)N)S",           # Cysteine
    "NC(CC(=O)O)C(=O)O",        # Asparagine
]
names = ["water", "alanine", "valine", "leucine", "isoleucine", "cysteine", "asparagine"]

estimate_gse_for_molecules(smiles_list, names)


Processing: water (O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -76.388251390 Eh  (conf 40)
Ground-state estimate (ANI): -76.388251390 Eh  from tautomer 0, conformer 40

Processing: alanine (N[C@@H](C)C(=O)O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -323.661033089 Eh  (conf 80)
Ground-state estimate (ANI): -323.661033089 Eh  from tautomer 0, conformer 80

Processing: valine (CC(C)C(N)C(=O)O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -402.271174230 Eh  (conf 106)
Ground-state estimate (ANI): -402.271174230 Eh  from tautomer 0, conformer 106

Processing: leucine (CC(C)CC(N)C(=O)O)
/Users/sanskriti/.pyenv/versions/3.10.12/envs/tf310/lib/python3.10/site-packages/torchani/resources/
[taut 0] lowest ANI energy: -441.575156956 Eh  (co

## 5) Getting 'true' values from QMPRot Database.
Theirs are calculated using the STO-3G database, so they are quite approximate...
However, this is a good way to get a starting point.

In [27]:
import pandas as pd

def get_qmprot_energies(names):
    """
    Loads energies for a list of amino acid names from the Pennylane dataset.
    Returns a pandas DataFrame with columns: name, abbreviation, energy.
    """
    records = []
    for name in names:
        try:
            data = qml.data.load("other", name=name, attributes=["abbreviation", "energy"])
            if data and len(data) > 0:
                records.append({
                    "name": name,
                    "abbreviation": getattr(data[0], "abbreviation", ""),
                    "energy": data[0].energy
                })
                print(f"{name.upper()} energy: {data[0].energy}")
            else:
                records.append({
                    "name": name,
                    "abbreviation": "",
                    "energy": None
                })
                print(f"{name.upper()}: No data found")
        except Exception as e:
            print(f"Error loading {name}: {e}")
            records.append({
                "name": name,
                "abbreviation": "",
                "energy": None
            })
    
    df = pd.DataFrame(records)
    return df

# Get reference energies for amino acids that match your SMILES list
amino_acids_subset = ["cys", "asn", "ala", "val", "leu", "ile"]
print("Loading QMPRot reference energies...")
df_energies = get_qmprot_energies(amino_acids_subset)
print("\nQMPRot Energy DataFrame:")
print(df_energies)

# Save to file for later comparison
df_energies.to_csv("qmprot_reference_energies.csv", index=False)
print("\nReference energies saved to 'qmprot_reference_energies.csv'")

Loading QMPRot reference energies...
CYS energy: -710.8573
ASN energy: -483.2392307
ALA energy: -317.69135
VAL energy: -394.84749
LEU energy: -433.42225
ILE energy: -432.82708

QMPRot Energy DataFrame:
  name abbreviation      energy
0  cys          cys -710.857300
1  asn          asn -483.239231
2  ala          ala -317.691350
3  val          val -394.847490
4  leu          leu -433.422250
5  ile          ile -432.827080

Reference energies saved to 'qmprot_reference_energies.csv'


In [ ]:
# ### sample code to just get one
# # from qmprot
# data_cys = qml.data.load("other", name="cys", attributes=["abbreviation", "energy"])
# print("Cysteine energy:", data_cys[0].energy)
# print(dir(data_cys[0]))

## 6) Comparison code

In [ ]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool
import numpy as np

def compare_classical_vs_qmprot():
    """
    Creates a Bokeh plot comparing classical ANI estimates vs QMPRot reference values
    """
    # Load the classical estimates from file
    classical_data = {}
    try:
        with open("estimations_gse.txt", "r") as f:
            for line in f:
                if line.strip() and not line.startswith("Name"):  # Skip header if exists
                    parts = line.strip().split("\t")
                    if len(parts) >= 3 and "No ANI result" not in parts[2]:
                        name = parts[0]
                        energy = float(parts[2].replace(" Eh", ""))
                        classical_data[name] = energy
    except FileNotFoundError:
        print("Run estimate_gse_for_molecules() first to generate estimations_gse.txt")
        return
    
    # Get QMPRot reference data
    amino_acid_names = ["cys", "asn", "ala", "val", "leu", "ile"]
    df_qmprot = get_qmprot_energies(amino_acid_names)
    
    # Create mapping from full names to abbreviations for matching
    name_mapping = {
        "cysteine": "cys",
        "asparagine": "asn", 
        "alanine": "ala",
        "valine": "val",
        "leucine": "leu",
        "isoleucine": "ile"
    }
    
    # Match up the data
    classical_vals = []
    qmprot_vals = []
    labels = []
    
    for full_name, abbrev in name_mapping.items():
        if full_name in classical_data:
            qmprot_row = df_qmprot[df_qmprot['name'] == abbrev]
            if not qmprot_row.empty and qmprot_row['energy'].iloc[0] is not None:
                classical_vals.append(classical_data[full_name])
                qmprot_vals.append(float(qmprot_row['energy'].iloc[0]))
                labels.append(full_name.capitalize())
    
    if not classical_vals:
        print("No matching data found between classical estimates and QMPRot values")
        return
    
    # Create Bokeh plot
    output_notebook()
    
    p = figure(
        width=600, height=600,
        title="Classical ANI vs QMPRot Ground State Energies",
        x_axis_label="ANI-2x Energy (Hartree)",
        y_axis_label="QMPRot Energy (Hartree)"
    )
    
    # Add scatter points
    scatter = p.circle(
        classical_vals, qmprot_vals, 
        size=10, color='blue', alpha=0.7,
        legend_label="Amino Acids"
    )
    
    # Add hover tool
    hover = HoverTool(
        tooltips=[
            ("Molecule", "@labels"),
            ("ANI-2x", "@x{0.000000}"),
            ("QMPRot", "@y{0.000000}"),
            ("Difference", "@diff{0.000000}")
        ],
        renderers=[scatter]
    )
    p.add_tools(hover)
    
    # Add perfect correlation line
    min_val = min(min(classical_vals), min(qmprot_vals))
    max_val = max(max(classical_vals), max(qmprot_vals))
    p.line([min_val, max_val], [min_val, max_val], 
           line_color='red', line_dash='dashed', 
           legend_label="Perfect Correlation")
    
    # Add data source for hover
    from bokeh.models import ColumnDataSource
    differences = [c - q for c, q in zip(classical_vals, qmprot_vals)]
    source = ColumnDataSource(data=dict(
        x=classical_vals,
        y=qmprot_vals,
        labels=labels,
        diff=differences
    ))
    
    # Update scatter to use source
    p.circle('x', 'y', size=10, color='blue', alpha=0.7, 
             source=source, legend_label="Amino Acids")
    
    p.legend.location = "top_left"
    
    # Print correlation statistics
    correlation = np.corrcoef(classical_vals, qmprot_vals)[0, 1]
    mae = np.mean(np.abs(differences))
    rmse = np.sqrt(np.mean(np.array(differences)**2))
    
    print(f"\nCorrelation Statistics:")
    print(f"Correlation coefficient: {correlation:.4f}")
    print(f"Mean Absolute Error: {mae:.6f} Hartree")
    print(f"Root Mean Square Error: {rmse:.6f} Hartree")
    print(f"Number of molecules: {len(classical_vals)}")
    
    show(p)
    
    return p

# Install bokeh if needed and run the comparison
try:
    import bokeh
except ImportError:
    print("Installing bokeh...")
    !pip install bokeh

# Run the comparison
compare_classical_vs_qmprot()

Found 7 classical estimates
✓ Loaded ala: -317.69135
✓ Loaded phe: -544.43743
✓ Loaded asp: -502.76713
✓ Loaded glu: -541.3498
✓ Loaded gly: -279.11151
✓ Loaded his: -538.52442
✓ Loaded ile: -432.82708
✓ Loaded leu: -433.42225
✓ Loaded lys: -487.7406
✓ Loaded met: -788.02138
✓ Loaded gln: -521.82178
✓ Loaded asn: -483.2392307
✓ Loaded pro: -393.7002
✓ Loaded cys: -710.8573
✗ Error loading ser: Unable to synchronously open file (truncated file: eof = 96, sblock->base_addr = 0, stored_eof = 2048)
✓ Loaded thr: -430.09637
✗ Error loading trp: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"
✗ Error loading tyr: "Unable to synchronously open object (object 'abbreviation' doesn't exist)"
✓ Loaded val: -394.84749
✓ Loaded arg: -595.17254
Found 17 QMPRot reference values
Matched alanine: ANI=-323.661033, QMPRot=-317.691350
Matched isoleucine: ANI=-402.271174, QMPRot=-432.827080
Matched leucine: ANI=-441.575157, QMPRot=-433.422250
Matched asparagine: ANI=-512.190860,

Loading BokehJS ...


Correlation Statistics:
Correlation coefficient: 0.9906
Mean Absolute Error: 15.337297 Hartree
Root Mean Square Error: 18.481607 Hartree
Number of molecules: 6


figure(id='p1364', ...)